# Test 02: Scala/Java-Only Installation

Integration test for the `sfnb_multilang` toolkit -- Scala/Java only.

**What this tests:**
1. Toolkit installation via pip
2. JDK + Scala installation (micromamba + coursier)
3. Snowpark JAR resolution
4. Ammonite REPL initialisation
5. `%%scala` magic registration and execution
6. Python <-> Scala data transfer
7. Snowpark Scala session (credential injection)
8. Optional: Spark Connect

**Prerequisites:**
- EAI with Maven Central + GitHub + PyPI access enabled on this notebook

## 1. Install the Toolkit

In [ ]:
# Install from local source (notebooks/ is inside the package dir)
!pip install -q ..

## 2. Install Scala/Java via Toolkit

This replaces `!bash setup_scala_environment.sh`. The toolkit:
1. Runs preflight checks (disk, network)
2. Installs micromamba + OpenJDK 17 (~15s)
3. Downloads coursier, resolves Snowpark + Scala + Ammonite JARs (~15s)
4. Installs JPype1 and writes metadata

**Expected time:** ~30s on fresh container, ~2s on re-run (cached).

In [ ]:
from sfnb_multilang import install

report = install(languages=["scala"])

In [ ]:
print(f"Success: {report.success}")
print(f"Env prefix: {report.env_prefix}")
print(f"Duration: {report.elapsed_seconds:.1f}s")

scala_result = report.plugin_results.get("scala")
if scala_result:
    print(f"Scala version: {scala_result.version}")
    print(f"Scala install OK: {scala_result.success}")

## 3. Verify Metadata File

The Scala plugin writes a JSON metadata file that the helper reads
for classpath configuration.

In [ ]:
import json, os

meta_path = os.path.expanduser("~/scala_jars/scala_env_metadata.json")
assert os.path.isfile(meta_path), f"FAIL: metadata not found at {meta_path}"

with open(meta_path) as f:
    meta = json.load(f)

for key in ["java_home", "scala_full_version", "snowpark_version", "ammonite_version"]:
    print(f"  {key}: {meta.get(key, 'MISSING')}")

print(f"\nPASS: Metadata file present and readable")

## 4. Set Up Scala Magic

Load the helper module and register `%%scala` / `%%java` magics.

In [ ]:
from scala_helpers import setup_scala_environment
setup_scala_environment()

## 5. Basic Scala Execution

In [ ]:
%%scala
val greeting = "Hello from Scala!"
println(greeting)
println(s"Scala version: ${scala.util.Properties.versionString}")
println(s"Java version: ${System.getProperty("java.version")}")

## 6. Python <-> Scala Variable Transfer

The `-i`/`-o` flags pass primitive values (int, float, string, bool)
between Python and Scala via JPype. For DataFrame transfer, use
Snowpark or PySpark DFs (see Section 7).

In [ ]:
py_count = 5
py_label = "test"
print(f"Sending to Scala: count={py_count}, label={py_label}")

In [ ]:
%%scala -i py_count,py_label -o scala_result
val n = py_count.asInstanceOf[Int]
val lbl = py_label.asInstanceOf[String]
val scala_result = (1 to n).map(i => s"$lbl-$i").mkString(", ")
println(s"PASS: Generated $n items: $scala_result")

## 7. Snowpark Scala Session

Test that Scala can access Snowflake via the Snowpark session
(credentials injected from the Python session).

In [ ]:
from snowflake.snowpark.context import get_active_session
from scala_helpers import bootstrap_snowpark_scala

session = get_active_session()
ok, msg = bootstrap_snowpark_scala(session)
print(f"Bootstrap: {'OK' if ok else 'FAIL'}")
if msg:
    print(msg)

In [ ]:
%%scala
val result = sfSession.sql("SELECT CURRENT_ACCOUNT() AS acct, CURRENT_ROLE() AS role").collect()
result.foreach(r => println(s"  Account: ${r.getString(0)}, Role: ${r.getString(1)}"))
println("PASS: Snowpark Scala session works")

## 8. Java Execution and Snowpark Java Session

The `%%java` magic runs on the same JVM. We also test Snowpark Java
connectivity using `javaSession` (bootstrapped from the Python session).

In [ ]:
%%java
System.out.println("Hello from Java!");
System.out.println("Java version: " + System.getProperty("java.version"));
System.out.println("PASS: Java execution works");

In [ ]:
from scala_helpers import bootstrap_snowpark_java
ok, msg = bootstrap_snowpark_java(session)
print(f"Java Snowpark bootstrap: {'OK' if ok else 'FAIL'}")
if msg:
    print(msg)

In [ ]:
%%java
Row[] rows = javaSession.sql("SELECT CURRENT_ACCOUNT() AS acct, CURRENT_ROLE() AS role").collect();
System.out.println("  Account: " + rows[0].getString(0));
System.out.println("  Role:    " + rows[0].getString(1));
System.out.println("PASS: Snowpark Java session works");

## 9. Spark Connect

Test Spark Connect integration. This re-installs with Spark Connect
enabled (adds `pyspark`, `snowpark-connect`, and Spark Connect client JARs),
then creates a Scala `spark` session over gRPC.

**Expected time:** ~40s for re-install (PySpark + client JARs), ~10s incremental if base cached.

In [ ]:
from sfnb_multilang import install

report = install(languages=["scala"], scala_spark_connect=True)
print(f"Re-install with Spark Connect: {'OK' if report.success else 'FAIL'}")
print(f"Duration: {report.elapsed_seconds:.1f}s")

In [ ]:
from scala_helpers import setup_scala_environment, setup_spark_connect

setup_scala_environment()
result = setup_spark_connect(session)
print(f"Spark Connect: {'OK' if result['success'] else 'FAIL'}")
if result.get("pyspark_version"):
    print(f"PySpark version: {result['pyspark_version']}")
if result.get("errors"):
    for e in result["errors"]:
        print(f"  Error: {e}")

In [ ]:
%%scala
spark.sql("SELECT 'hello' AS msg, 42 AS num").show()
println("PASS: Spark Connect works")

## Test Summary

| Test | Expected |
|---|---|
| Toolkit install | pip succeeds |
| Scala install | report.success == True |
| Metadata | JSON file present with versions |
| Scala magic | `%%scala` cells execute |
| Data transfer | Python <-> Scala round-trip via -i/-o |
| Snowpark Scala | sfSession connects to Snowflake |
| Java magic | `%%java` cells execute |
| Snowpark Java | javaSession connects to Snowflake |
| Spark Connect | Re-install, gRPC server starts, spark.sql works |